# RAG Retrieval Layer
### Ein geführter Workshop zur vierten Online-Stufe: von persistierten Indizes zu erklärten Ergebnissen

## Anschluss an die Indexing Layer

Die Indexing Layer ist abgeschlossen. Ihr Ergebnis sind drei persistierte Artefakte:

- `documents.jsonl`: der DocumentStore mit Text, Metadaten und IDs als Single Source of Truth
- Dense-Index-Dateien: BruteForce- oder FAISS-Backend mit echten Embedding-Vektoren
- Sparse-Index-Dateien: der invertierte Index mit Termen, TF-Statistiken und Vokabular

Die Retrieval Layer beginnt genau an dieser Grenze. Sie berechnet keine Embeddings offline,
baut keine Indizes, transformiert keine Chunks. Sie liest ausschließlich persistierte
Artefakte und beantwortet damit Queries zur Laufzeit. Eine Query durchläuft dabei diese
Schritte:

1. Query-Transformation: Dense-Embedding und Tokenisierung
2. Candidate Generation: Kandidaten aus DenseIndex und SparseIndex
3. Scoring: BM25, TF-IDF oder Skalarprodukt
4. Fusion: Zusammenführen beider Wege (nur im Hybrid-Modus)
5. DocumentStore-Join: Text zu den Kandidaten nachladen
6. Reranking: optionale Nachbewertung
7. Ergebnis: eine geordnete Liste von `RetrievalResult`

Damit ist Retrieval die erste und einzige **Online-Stufe** nach dem Index-Aufbau.

### Offline vs. Online: Die entscheidende Grenze

| Layer | Betrieb | Ausgabe |
|---|---|---|
| Ingestion | offline | `chunks.jsonl` |
| Embedding | offline | `embeddings.jsonl`, echte Modellvektoren |
| Indexing | offline | `DenseIndex`, `SparseIndex`, `DocumentStore` |
| **Retrieval** | **online** | `RetrievalResult` pro Query |

### Architektonische Vorbedingung dieses Notebooks

Dieses Notebook verwendet **ausschließlich echte Artefakte**:

- Dense-Vektoren aus der Indexing Layer (erzeugt vom echten Embedding-Modell)
- Query-Vektoren vom echten Embedder (dieselbe Konfiguration wie beim Index-Aufbau)
- kein synthetisches Embedding, kein hash-basierter Fake-Vektor

Der Grund ist die Dimension: Der DenseIndex legt die Vektordimension fest, und jeder
Query-Vektor muss exakt dieselbe Dimension haben. Ein synthetischer 8-dimensionaler Vektor
gegen einen Index mit 1024-dimensionalen BGE-M3-Vektoren erzeugt unmittelbar einen
`ValueError`. Das Notebook bricht kontrolliert ab, wenn Artefakte fehlen oder Dimensionen
inkonsistent sind.


## Welche Bausteine dieses Notebook verwendet

Diese Stufe nutzt vor allem das Paket `rag.retrieval` und lädt dazu die Artefakte der
Indexing Layer sowie den Embedder aus der Embedding Layer. Überblick über die Module und
ihre Rolle:

**Artefakte und Embedder aus den vorherigen Layern.**
`rag.indexing.orchestrator.IndexingOrchestrator` und `rag.indexing.config` laden den
DenseIndex, den SparseIndex und den DocumentStore.
`rag.indexing.sparse.tokenizer.Tokenizer` und `...sparse.view.InvertedIndexView` liefern die
Query-Tokenisierung und die Korpusstatistiken.
`rag.embedding.*` (Config, Factory, Cache) erzeugt die Query-Vektoren mit derselben
Konfiguration wie beim Index-Aufbau.

**Die fünf Retrieval-Strategien.**
`rag.retrieval.dense.DenseRetriever` (semantische Suche),
`rag.retrieval.bm25.BM25Retriever` und `...tfidf.TFIDFRetriever` (lexikalische Suche),
`rag.retrieval.learned_sparse.LearnedSparseRetriever` (gelernte Sparse-Vektoren) und
`rag.retrieval.hybrid.HybridRetriever` (Kombination aus Dense und Sparse).

**Fusion, Reranking und Steuerung.**
`rag.retrieval.fusion.fuse` führt zwei Ergebnislisten zusammen.
`rag.retrieval.reranker.BaselineReranker` bewertet die oberen Ränge nach.
`rag.retrieval.orchestrator.RetrievalOrchestrator` verbindet alle Schritte zu einer Pipeline
und kann über `retrieve_with_trace()` jeden Zwischenschritt offenlegen.

**Scoring-Funktionen und Datentypen.**
`rag.retrieval.scoring` (`cosine_similarity`, `dot_product`, `min_max_normalise`, `bm25_idf`,
`bm25_tf_component`, `sparse_dot_product`) enthält die einzelnen Rechenbausteine.
`rag.retrieval.config` und `rag.retrieval.types` definieren die Konfigurationen und die
typisierten Ergebnisobjekte (`ScoredCandidate`, `RankedCandidate`, `RetrievalResult`,
`RetrievalTrace`).

Dazu `pathlib`, `math` und `logging` aus der Standardbibliothek. Fast alles hier ist
austauschbar und einzeln aufrufbar: Jede Strategie lässt sich isoliert ausführen, die
Fusion zwischen gewichteter Summe und Reciprocal Rank Fusion umschalten, das Reranking
ein- und ausschalten, und der Modus zwischen rein dense, rein sparse und hybrid wählen.


---

## 1. Persistierte Artefakte laden

In [ ]:
from pathlib import Path
import math
import logging

logging.getLogger("rag.indexing").setLevel(logging.ERROR)
logging.getLogger("rag.retrieval").setLevel(logging.ERROR)
logging.getLogger("rag.embedding").setLevel(logging.ERROR)

index_base   = Path("/home/jovyan/workspace/data/index")
persist_path = index_base / "persist_demo"

required_paths = {
    "persist_demo/":        persist_path,
    "documents.jsonl":      persist_path / "documents.jsonl",
}

print("Artefakt-Pruefung:")
all_present = True
for label, path in required_paths.items():
    exists   = path.exists()
    size     = f"{path.stat().st_size / 1024:.1f} KB" if exists and path.is_file() else ("-" if not exists else "dir")
    print(f"  {'OK' if exists else 'FEHLT':<5}  {label:<28}  {size}")
    if not exists:
        all_present = False

if not all_present:
    raise RuntimeError(
        "Indexing-Artefakte fehlen. "
        "Bitte zuerst das Indexing-Notebook vollstaendig ausfuehren (Abschnitt 16, persist=True). "
        "Das Retrieval-Notebook erzeugt keine Demo-Daten."
    )

print()
print("Alle erforderlichen Artefakte vorhanden.")

In [ ]:
from rag.indexing.orchestrator import IndexingOrchestrator
from rag.indexing.config import IndexConfig, DenseIndexConfig, SparseIndexConfig

cfg = IndexConfig(
    index_dir=persist_path,
    mode="hybrid",
    dense=DenseIndexConfig(backend="brute_force", metric="cosine"),
    sparse=SparseIndexConfig(tokenizer="simple"),
)

index_result = IndexingOrchestrator(cfg).load()

dense_index  = index_result.dense_index
sparse_index = index_result.sparse_index
doc_store    = index_result.document_store

all_docs  = list(doc_store.stream_all())
doc_by_id = doc_store.load_by_id()

print(f"DenseIndex     : {type(dense_index).__name__}")
print(f"SparseIndex    : {type(sparse_index).__name__}")
print(f"DocumentStore  : {len(all_docs)} Dokumente")

---

## 2. Dimensionsvalidierung

Der Dense-Index legt die Vektordimension fest. Diese Dimension wird zur Laufzeit
für jeden Query-Vektor erzwungen.

**Invariante:** Alle Dense-Vektoren im DocumentStore müssen dieselbe Dimension haben.
Inkonsistenz deutet auf einen fehlerhaften Index-Build hin.

In [ ]:
docs_with_dense = [d for d in all_docs if d.get("dense_vector")]

if not docs_with_dense:
    raise RuntimeError(
        "DocumentStore enthaelt keine Dokumente mit 'dense_vector'. "
        "Bitte das Indexing-Notebook im Hybrid- oder Dense-Modus ausfuehren."
    )

dense_dims = {len(d["dense_vector"]) for d in docs_with_dense}

if len(dense_dims) != 1:
    raise RuntimeError(
        f"Inkonsistente Dense-Dimensionen im DocumentStore: {dense_dims}. "
        "Der Index-Build war fehlerhaft."
    )

DENSE_DIM = next(iter(dense_dims))

sample_doc  = docs_with_dense[0]
sample_vec  = sample_doc["dense_vector"]
sample_norm = math.sqrt(sum(x * x for x in sample_vec))

print(f"Dense-Dimension : {DENSE_DIM}")
print(f"Dokumente (dense): {len(docs_with_dense)} / {len(all_docs)}")
print(f"Normierungsprobe : L2={sample_norm:.5f}  (sollte ≈ 1.0 bei normalisierten Vektoren)")
print(f"Vorschau         : {[round(x, 4) for x in sample_vec[:4]]}...")
print()
print("Diese Dimension ist bindend fuer alle Query-Vektoren in diesem Notebook.")
print("Ein Query-Vektor anderer Dimension fuehrt zu einem sofortigen ValueError.")

---

## 3. Tokenizer und Sparse-Index-View

Der Tokenizer muss exakt derselbe sein wie beim Index-Aufbau (Indexing-Notebook).

In [ ]:
from rag.indexing.sparse.tokenizer import Tokenizer
from rag.indexing.sparse.view import InvertedIndexView

tokenizer = Tokenizer(SparseIndexConfig(tokenizer="simple"))
view: InvertedIndexView = sparse_index.get_view()
stats = view.get_stats()

print(f"Tokenizer      : simple (identisch zum Index-Build)")
print(f"Vocabulary     : {len(view.get_vocabulary())} Terme")
print(f"doc_count      : {stats['doc_count']}")
print(f"avg_doc_length : {stats['avg_doc_length']:.2f} Tokens")
print()

if stats["doc_count"] != len(all_docs):
    raise RuntimeError(
        f"Inkonsistenz: SparseIndex doc_count={stats['doc_count']} "
        f"!= DocumentStore count={len(all_docs)}."
    )
print("Konsistenz-Check SparseIndex <-> DocumentStore: OK")

---

## 4. Echten Embedder laden

Der Embedder wird für Dense-Query-Vektoren benötigt: immer wenn der Orchestrator
im `dense`- oder `hybrid`-Modus arbeitet.

**Architektonische Anforderung:** Der Embedder muss dieselbe Konfiguration verwenden
wie beim Index-Aufbau (Embedding-Notebook). Insbesondere gleiche Modell-Klasse und
gleiche Output-Dimension.

Wenn der Embedder nicht verfügbar ist, stehen nur Sparse-Modi zur Verfügung.
Alle Dense- und Hybrid-Abschnitte werden dann konditionell übersprungen.

In [ ]:
from rag.embedding.config import EmbeddingConfig, EmbeddingBehaviorConfig, EmbeddingProjectionConfig
from rag.embedding.factory import create_embedder
from rag.embedding.model_cache import get_default_cache

embedding_config = EmbeddingConfig(
    provider="bge-m3",
    model_name="BAAI/bge-m3",
    model_version="1.0",
    device="cpu",
    batch_size=1,
    use_fp16=False,
    retrieval_mode="dense",
    behavior=EmbeddingBehaviorConfig(normalize=True, mode="query"),
    projection=EmbeddingProjectionConfig(target_dim=None, method="truncate", model_aware=True),
)

EMBEDDER_AVAILABLE = False
embedder = None

try:
    embedder = create_embedder(embedding_config, cache=get_default_cache())
    EMBEDDER_AVAILABLE = True
    print(f"Embedder geladen  : {embedder.model_type()}")
    print(f"Dimension         : {embedder.dimension()}")
    print(f"Supported modes   : {sorted(embedder.supported_modes())}")
except Exception as exc:
    print(f"Embedder nicht verfuegbar: {exc}")
    print()
    print("Nur Sparse-Modi (BM25, TF-IDF) sind ohne Embedder verfuegbar.")

In [ ]:
# Dimensionskonsistenz: Embedder <-> Dense-Index
if EMBEDDER_AVAILABLE:
    emb_dim = embedder.dimension()
    if emb_dim is not None and emb_dim != DENSE_DIM:
        raise RuntimeError(
            f"Dimensionsinkonsistenz: Embedder={emb_dim}, DenseIndex={DENSE_DIM}. "
            f"Der Embedder muss exakt dieselbe Konfiguration wie beim Index-Build verwenden."
        )
    print(f"Dimensionskonsistenz: Embedder ({emb_dim}) <-> DenseIndex ({DENSE_DIM}) -> OK")
else:
    print(f"Dimensionskonsistenz: uebersprungen (kein Embedder)")
    print(f"Dense-Index-Dimension aus gespeicherten Vektoren: {DENSE_DIM}")

In [ ]:
# Hilfsfunktion: echtes Query-Embedding mit Dimensionscheck
def embed_query(text: str) -> list:
    """Erzeuge echten Dense-Query-Vektor. Bricht ab wenn Embedder fehlt."""
    if not EMBEDDER_AVAILABLE:
        raise RuntimeError("Embedder nicht verfuegbar. Dense-Retrieval erfordert den echten Embedder.")
    vecs = embedder.embed_queries([text])
    if not vecs or not vecs[0]:
        raise RuntimeError(f"Embedder lieferte keinen Vektor fuer: '{text}'")
    vec = vecs[0]
    if len(vec) != DENSE_DIM:
        raise RuntimeError(
            f"Dimensionsfehler: Query-Vektor hat dim={len(vec)}, "
            f"DenseIndex erwartet dim={DENSE_DIM}."
        )
    return vec

if EMBEDDER_AVAILABLE:
    _probe_vec  = embed_query("Dense Retrieval Vektorsimilaritaet")
    _probe_norm = math.sqrt(sum(x * x for x in _probe_vec))
    print(f"Query-Embedding-Probe:")
    print(f"  Dim    : {len(_probe_vec)}")
    print(f"  L2-Norm: {_probe_norm:.5f}  (≈ 1.0 bei normalisiertem Embedder)")
    print(f"  Vorschau: {[round(x, 4) for x in _probe_vec[:4]]}...")
else:
    print("embed_query() definiert, aber Embedder nicht verfuegbar.")

### Proxy-Vektor für Retriever-Isolations-Tests

Für Abschnitte, die den DenseRetriever in Isolation demonstrieren, wird der
Dense-Vektor eines bereits indizierten Dokuments als Query verwendet.

**Warum das architektonisch korrekt ist:**
- Der Vektor ist real: er wurde vom echten Embedder bei der Indexierung erzeugt
- Er hat exakt die korrekte Dimension (DENSE_DIM)
- Er ist L2-normalisiert (identisch wie Query-Vektoren)
- Er erlaubt nachvollziehbare Experimente (perfekte Selbstähnlichkeit = Score ≈ 1.0)

Das ist kein Fake-Embedding. Es ist ein echter, persistierter Vektor aus dem Index.

In [ ]:
# Proxy-Query: echter Vektor aus dem DocumentStore (für Retriever-Isolation)
proxy_doc = docs_with_dense[0]
query_vec = list(proxy_doc["dense_vector"])

print(f"Proxy-Query-Vektor aus Dokument: {proxy_doc['id']}")
print(f"Dimension : {len(query_vec)}  (= DENSE_DIM = {DENSE_DIM})")
print(f"Text      : {proxy_doc['text'][:70]!r}...")
print()
print("Dieser Vektor wird in den Retriever-Isolations-Tests als Query verwendet.")
print("Fuer den vollstaendigen Orchestrator wird der echte Embedder aufgerufen.")

---

## 5. Die Datentypen der Retrieval Layer

| Typ | Inhalt | Entsteht nach |
|---|---|---|
| `ScoredCandidate` | `id` + `score` | Candidate Generation |
| `RankedCandidate` | `id` + `score` + `dense_score` + `sparse_score` | Fusion |
| `RetrievalResult` | vollständiger Record mit Text + Metadaten | DocumentStore-Join |
| `RetrievalTrace` | alle Intermediate States einer Pipeline-Ausführung | `retrieve_with_trace()` |

Die Trennung erzwingt, dass Text erst nach dem Join vorhanden ist.
Kein Retriever kann versehentlich auf Dokumenttext zugreifen. Er bekommt ihn nie.

In [ ]:
from rag.retrieval.types import (
    ScoredCandidate, RankedCandidate, RetrievalResult, RetrievalTrace, HybridQuery,
)

scored: ScoredCandidate = {"id": proxy_doc["id"], "score": 0.872}

ranked: RankedCandidate = {
    "id":           proxy_doc["id"],
    "score":        0.731,
    "dense_score":  0.872,
    "sparse_score": 3.14,
}

result_example: RetrievalResult = {
    "id":               proxy_doc["id"],
    "chunk_id":         proxy_doc["chunk_id"],
    "document_id":      proxy_doc["document_id"],
    "score":            0.731,
    "retrieval_score":  0.731,
    "text":             proxy_doc["text"],
    "metadata":         proxy_doc["metadata"],
}

print(f"{'Typ':<28}  text  score  dense_score")
for name, obj in [
    ("ScoredCandidate",  scored),
    ("RankedCandidate",  ranked),
    ("RetrievalResult",  result_example),
]:
    print(f"  {name:<26}  {str('text' in obj):<5}  {str('score' in obj):<5}  {str('dense_score' in obj):<5}")
print()
print("Invariante: Nur RetrievalResult enthaelt Text. Alle anderen Types sind textfrei.")

## 6. BaseRetriever: Der gemeinsame Vertrag

Alle fünf Retrieval-Strategien implementieren `BaseRetriever`:

```python
class BaseRetriever(ABC):
    @abstractmethod
    def retrieve_candidates(self, query: Any, k: int) -> List[ScoredCandidate]:
        ...
```

**Invarianten aller Implementierungen:**
- Sie geben ausschließlich `(id, score)` zurück, niemals Text.
- Ein ungültiger Query-Typ ergibt eine leere Liste (keine Exception).
- Ein leerer Query ergibt eine leere Liste.
- `k <= 0` ergibt eine leere Liste.
- Das Ergebnis ist deterministisch sortiert: Score absteigend, bei Gleichstand `id` aufsteigend.

**Polymorphe Query-Typen:** Jede Strategie erwartet einen anderen Query-Typ.

| Strategie | Query-Typ | Bedeutung |
|---|---|---|
| `DenseRetriever` | `List[float]` | echter Embedding-Vektor |
| `BM25Retriever` | `List[str]` oder `str` | tokenisierter Text |
| `TFIDFRetriever` | `List[str]` oder `str` | tokenisierter Text |
| `LearnedSparseRetriever` | `Dict[str, float]` | gelernter Sparse-Vektor |
| `HybridRetriever` | `HybridQuery` | dict mit `"dense"` und/oder `"sparse"` |


---

## 7. Dense Retrieval (isoliert)

`DenseRetriever` umhüllt einen `BaseDenseIndex` und übersetzt dessen Ausgabe in
`List[ScoredCandidate]`.

Als Query wird `query_vec` verwendet: ein echter persistierter Embedding-Vektor
aus dem DocumentStore, der die korrekte Dimension `DENSE_DIM` hat.

In [ ]:
from rag.retrieval.dense import DenseRetriever
from rag.retrieval.config import DenseRetrievalConfig

dense_config    = DenseRetrievalConfig(candidate_k=20)
dense_retriever = DenseRetriever(dense_index=dense_index, config=dense_config)

dense_candidates = dense_retriever.retrieve_candidates(query=query_vec, k=5)

print(f"Query-Dim      : {len(query_vec)}")
print(f"DENSE_DIM      : {DENSE_DIM}")
print(f"Returned       : {len(dense_candidates)} ScoredCandidate")
print()
for rank, c in enumerate(dense_candidates, 1):
    doc          = doc_by_id.get(c["id"])
    text_preview = doc["text"][:50] if doc else "?"
    print(f"  {rank}. score={c['score']:+.5f}  {text_preview!r}...")
print()
assert all("text" not in c for c in dense_candidates), "INVARIANT VERLETZT"
print("Invariante bestaetigt: kein 'text'-Feld in ScoredCandidate.")

### Score-Semantik: cosine, dot, l2

| Metrik | Wertebereich | Bedeutung |
|---|---|---|
| `cosine` | [-1, 1] | Winkelähnlichkeit; 1.0 = identische Richtung |
| `dot` | unbeschränkt | Skalarprodukt; bei normalisierten Vektoren = cosine |
| `l2` | (-∞, 0] | negierte euklidische Distanz; 0.0 = identisch |

In [ ]:
from rag.retrieval.scoring import cosine_similarity, dot_product

vec_a = list(docs_with_dense[0]["dense_vector"])
vec_b = list(docs_with_dense[min(1, len(docs_with_dense)-1)]["dense_vector"])

cos_same = cosine_similarity(vec_a, vec_a)
cos_diff = cosine_similarity(vec_a, vec_b)
dot_same = dot_product(vec_a, vec_a)
dot_diff = dot_product(vec_a, vec_b)

print("Scoring-Experiment: identischer vs. verschiedener Vektor")
print()
print(f"cosine(a, a) = {cos_same:+.5f}   (Selbst-Aehnlichkeit = 1.0)")
print(f"cosine(a, b) = {cos_diff:+.5f}   (andere Richtung)")
print(f"dot(a, a)    = {dot_same:+.5f}   (fuer L2-normalisierte Vektoren = cosine)")
print(f"dot(a, b)    = {dot_diff:+.5f}")
print()
diff = abs(cos_diff - dot_diff)
print(f"|cosine - dot| = {diff:.2e}  (Rundungsfehler, erwartet ≈ 0)")

---

## 8. Scoring-Funktion: min_max_normalise

`min_max_normalise` wird in der Weighted-Sum-Fusion für per-Leg-Normalisierung verwendet.

**Spezialfall gleiche Scores:**
Wenn alle Scores identisch sind, gibt `min_max_normalise` überall `1.0` zurück.
Ein Retrieval-Leg mit uniformen Scores trägt ein valides Signal: alle Kandidaten
sind gleichwertig. Würde man `0.0` zurückgeben, würde das Leg still aus der
Fusion herausfallen.

In [ ]:
from rag.retrieval.scoring import min_max_normalise

cases = [
    ("verschieden", [2.5, 1.0, 3.0, 0.5, 2.0]),
    ("alle gleich",  [1.7, 1.7, 1.7]),
    ("einzelner",    [4.2]),
]

print("min_max_normalise, Spezialfaelle:")
print()
for label, scores in cases:
    result = min_max_normalise(scores)
    print(f"  Eingabe ({label:<11}): {scores}")
    print(f"  Ausgabe             : {[round(x, 3) for x in result]}")
    print()
print("Alle gleich -> 1.0 (nicht 0.0): das Leg bleibt in der Fusion aktiv.")

## 9. Sparse Retrieval: BM25 (isoliert)

`BM25Retriever` implementiert das probabilistische Rankingmodell BM25.

**Ablauf:**
1. Query als `str` wird tokenisiert (oder direkt als `List[str]` übergeben)
2. `SparseIndex.query()` liefert ungewichtete Kandidaten-IDs (ohne Score)
3. die Korpusstatistiken kommen aus der `InvertedIndexView`
4. BM25 berechnet für jeden Kandidaten einen Score
5. absteigend sortieren und auf top-k kürzen ergibt die `ScoredCandidate`

**Warum der SparseIndex nicht selbst scored:** Der BM25-IDF-Anteil hängt von der Query ab.
Ein Index, der bereits zur Build-Zeit scoren würde, könnte keine query-abhängigen Scores
liefern. `SparseIndex.query()` gibt deshalb nur IDs zurück, den Score berechnet der Retriever.

**BM25-Formel:**
```
IDF(t)      = log((N - df(t) + 0.5) / (df(t) + 0.5) + 1)
TF_sat(t,d) = (tf * (k1+1)) / (tf + k1*(1 - b + b*(dl/avgdl)))
score(q,d)  = Σ_t IDF(t) * TF_sat(t,d)
```

**Zum Experimentieren:** `k1` steuert die TF-Sättigung (wie stark häufige Terme noch
zählen), `b` die Längennormalisierung (wie stark lange Dokumente abgewertet werden). Höheres
`k1` gewichtet wiederholte Terme stärker, `b=0` schaltet die Längennormalisierung ganz ab.
In Produktivsystemen sind `k1` zwischen 1.2 und 2.0 und `b` um 0.75 übliche Startwerte.


In [ ]:
from rag.retrieval.bm25 import BM25Retriever
from rag.retrieval.config import BM25Config

bm25_config    = BM25Config(k1=1.5, b=0.75, candidate_k=20)
bm25_retriever = BM25Retriever(
    sparse_index=sparse_index,
    view=view,
    tokenizer=tokenizer,
    config=bm25_config,
)

query_text      = "Retrieval Augmented Generation Sprachmodell"
bm25_candidates = bm25_retriever.retrieve_candidates(query=query_text, k=5)

print(f"Query  : '{query_text}'")
print(f"Tokens : {tokenizer.tokenize(query_text)}")
print()
print(f"BM25 Kandidaten (top-{len(bm25_candidates)}):")
for rank, c in enumerate(bm25_candidates, 1):
    doc  = doc_by_id.get(c["id"])
    text = doc["text"][:55] if doc else "?"
    print(f"  {rank}. score={c['score']:.4f}  {text!r}...")

### BM25 Intermediate States: Scoring sichtbar machen

BM25 kombiniert IDF und TF-Sättigung. Der folgende Block zeigt die Einzelkomponenten
für den Top-Kandidaten. Es ist exakt das, was `BM25Retriever` intern berechnet.

In [ ]:
from rag.retrieval.scoring import bm25_idf, bm25_tf_component

query_tokens = tokenizer.tokenize(query_text)
stats_snap   = view.get_stats()
n_docs       = stats_snap["doc_count"]
avg_dl       = stats_snap["avg_doc_length"]
df_map       = stats_snap["document_frequency"]
dl_map       = stats_snap["doc_lengths"]

if bm25_candidates:
    cid = bm25_candidates[0]["id"]
    dl  = dl_map.get(cid, 0)

    print(f"Kandidat    : {cid}")
    print(f"doc_length  : {dl} Tokens")
    print(f"avg_dl      : {avg_dl:.2f}")
    print()
    print(f"{'Token':<18} {'tf':>4}  {'df':>4}  {'IDF':>7}  {'TF_sat':>7}  {'beitrag':>8}")
    print("-" * 58)

    total = 0.0
    for token in sorted(set(query_tokens)):
        tf      = view.get_tf(token, cid)
        df      = df_map.get(token, 0)
        idf     = bm25_idf(df=df, n=n_docs)
        tf_comp = bm25_tf_component(tf=tf, doc_length=dl, avg_doc_length=avg_dl,
                                    k1=bm25_config.k1, b=bm25_config.b)
        contrib = idf * tf_comp
        total  += contrib
        print(f"  {token:<16} {tf:>4}  {df:>4}  {idf:>7.4f}  {tf_comp:>7.4f}  {contrib:>8.4f}")
    print("-" * 58)
    print(f"  {'BM25 score':>43}  {total:>8.4f}")
    print()
    print(f"  BM25Retriever liefert: {bm25_candidates[0]['score']:.4f}  (identisch)")
else:
    print("Keine BM25-Kandidaten fuer diese Query.")

---

## 10. Sparse Retrieval: TF-IDF (isoliert)

| Eigenschaft | TF-IDF | BM25 |
|---|---|---|
| TF-Sättigung | nein | ja (`k1`-Parameter) |
| Längennormalisierung | nein | ja (`b`-Parameter) |
| IDF | `log((N+1)/(df+1)) + 1` | `log((N-df+0.5)/(df+0.5) + 1)` |

Bei variierenden Dokumentlängen setzt BM25 relevantere Gewichte als TF-IDF.

In [ ]:
from rag.retrieval.tfidf import TFIDFRetriever
from rag.retrieval.config import TFIDFConfig

tfidf_config    = TFIDFConfig(sublinear_tf=True, candidate_k=20)
tfidf_retriever = TFIDFRetriever(
    sparse_index=sparse_index,
    view=view,
    tokenizer=tokenizer,
    config=tfidf_config,
)

tfidf_candidates = tfidf_retriever.retrieve_candidates(query=query_text, k=5)

bm25_by_id = {c["id"]: c["score"] for c in bm25_candidates}
print(f"Query: '{query_text}'")
print()
print(f"{'Rang':<5} {'TF-IDF':>10}  {'BM25':>10}  Text[:40]")
print("-" * 68)
for rank, c in enumerate(tfidf_candidates, 1):
    doc   = doc_by_id.get(c["id"])
    bm25s = bm25_by_id.get(c["id"], float("nan"))
    text  = doc["text"][:40] if doc else "?"
    print(f"  {rank:<3}  {c['score']:>10.4f}  {bm25s:>10.4f}  {text!r}...")

---

## 11. Kontrolliertes Experiment: BM25 vs. TF-IDF

**Hypothese:** Bei kurzen, gleichlangen Texten divergieren die Rankings kaum.
Bei Texten mit stark verschiedener Länge bevorzugt BM25 kürzere Dokumente
(Längennormalisierung via `b`-Parameter), TF-IDF ist längenneutral.

In [ ]:
test_queries = [
    "Transformer Encoder Decoder",
    "Retrieval Augmented Generation",
    "BM25 Ranking probabilistisch",
    "Dense Retrieval Vektorsimilaritaet",
]

print(f"{'Query':<38}  BM25-top3                TF-IDF-top3")
print("-" * 90)
for q in test_queries:
    bm25_r  = bm25_retriever.retrieve_candidates(query=q, k=5)
    tfidf_r = tfidf_retriever.retrieve_candidates(query=q, k=5)
    bm25_ids  = [c["id"].split("-")[-1] for c in bm25_r[:3]]
    tfidf_ids = [c["id"].split("-")[-1] for c in tfidf_r[:3]]
    match = "==" if bm25_ids == tfidf_ids else "!="
    print(f"  {q:<36}  {str(bm25_ids):<24}  {match}  {tfidf_ids}")

---

## 12. Learned Sparse Retrieval (isoliert)

`LearnedSparseRetriever` unterscheidet sich fundamental von BM25 und TF-IDF:

- Keine Korpusstatistiken (kein IDF, kein `avg_doc_length`)
- Scoring via **Sparse Dot Product** über Token-Gewichts-Dicts
- Token-IDs sind Vokabular-IDs des Embedding-Modells (z.B. BGE-M3 Lexical Weights)

```
score(q, d) = Σ_{t ∈ q ∩ d} q[t] * d[t]
```

**Voraussetzung:** Dokumente müssen `sparse_vector`-Felder enthalten.
Das erfordert Hybrid- oder Sparse-Embedding-Modus im Embedding-Notebook.

In [ ]:
from rag.retrieval.learned_sparse import LearnedSparseRetriever
from rag.retrieval.config import LearnedSparseConfig
from rag.retrieval.scoring import sparse_dot_product

docs_with_sparse = [d for d in all_docs if d.get("sparse_vector")]
print(f"Dokumente mit sparse_vector: {len(docs_with_sparse)} / {len(all_docs)}")
print()

if not docs_with_sparse:
    print("HINWEIS: Keine Sparse-Vektoren im DocumentStore.")
    print("Learned Sparse Retrieval erfordert Hybrid-Embedding-Modus.")
    print("Das Embedding-Notebook muss mit retrieval_mode='hybrid' ausgefuehrt worden sein.")
    LEARNED_SPARSE_AVAILABLE = False
else:
    LEARNED_SPARSE_AVAILABLE = True
    sample_sparse_doc = docs_with_sparse[0]
    sample_sparse_vec = sample_sparse_doc["sparse_vector"]
    print(f"Beispiel Sparse-Vektor (Dokument: {sample_sparse_doc['id']}):")
    items = list(sample_sparse_vec.items())[:6]
    for token_id, weight in items:
        print(f"  token_id={token_id:<8}  weight={weight:.4f}")
    print(f"  ... ({len(sample_sparse_vec)} tokens total)")

In [ ]:
if LEARNED_SPARSE_AVAILABLE:
    learned_config    = LearnedSparseConfig(candidate_k=10, min_score=0.0, use_index_pruning=True)
    learned_retriever = LearnedSparseRetriever(document_store=doc_store, config=learned_config)

    query_sparse_vec  = dict(sample_sparse_vec)
    learned_candidates = learned_retriever.retrieve_candidates(query=query_sparse_vec, k=5)

    print(f"Query-Vektor (Proxy aus Dokument '{sample_sparse_doc['id']}'):")
    print(f"  Tokens im Query-Vektor: {len(query_sparse_vec)}")
    print(f"  Kandidaten mit Ueberlappung: {len(learned_candidates)}")
    print()
    for rank, c in enumerate(learned_candidates, 1):
        doc     = doc_by_id.get(c["id"])
        dsv     = doc.get("sparse_vector", {}) if doc else {}
        overlap = set(query_sparse_vec) & set(dsv)
        print(f"  {rank}. score={c['score']:.4f}  overlap={len(overlap)} tokens")
else:
    print("Learned Sparse: uebersprungen (keine Sparse-Vektoren im Index).")
    learned_retriever  = None
    learned_candidates = []

## 13. Candidate Pool Design: Warum candidate_k > top_k

`candidate_k` bestimmt, wie viele Kandidaten intern abgefragt werden, bevor auf `top_k`
gekürzt wird.

**Das Problem:** Bei `candidate_k = top_k = 5` kann Fusion oder Reranking nur unter 5
Kandidaten wählen. Ein Dokument auf Rang 7, das nach Fusion vielleicht Rang 2 erreichen würde,
wird nie gesehen.

**Die Lösung:** `retrieve_candidates()` fragt intern `max(k, candidate_k)` ab. Fusion und
Reranking arbeiten auf dem größeren Pool.

**Zum Experimentieren:** Die nächste Zelle vergleicht ein enges und ein weites `candidate_k`.
Beim `BruteForceIndex` ist die Suche exakt, ein größeres `candidate_k` ändert die Top-3
deshalb nicht. Bei approximierten Indizes (FAISS IVF, HNSW) macht ein größeres `candidate_k`
einen echten Recall-Unterschied, weil mehr Kandidaten überhaupt erst in Reichweite kommen. In
Produktivsystemen wählt man `candidate_k` meist deutlich größer als `top_k`, damit Fusion und
Reranking genug Auswahl haben.


In [ ]:
dense_narrow = DenseRetriever(dense_index=dense_index, config=DenseRetrievalConfig(candidate_k=3))
dense_wide   = DenseRetriever(dense_index=dense_index, config=DenseRetrievalConfig(candidate_k=len(all_docs)))

cands_narrow = dense_narrow.retrieve_candidates(query=query_vec, k=3)
cands_wide   = dense_wide.retrieve_candidates(query=query_vec, k=3)

ids_narrow = {c["id"] for c in cands_narrow}
ids_wide   = {c["id"] for c in cands_wide}

print(f"candidate_k Experiment (final k=3, Index-Groesse={len(all_docs)})")
print()
print(f"candidate_k=3  :  IDs={sorted(c.split('-')[-1] for c in ids_narrow)}")
print(f"candidate_k=all:  IDs={sorted(c.split('-')[-1] for c in ids_wide)}")
print()
print(f"Ergebnisse identisch: {ids_narrow == ids_wide}")
print()
print("BruteForceIndex: exakte Suche, candidate_k aendert die Top-3 nicht.")
print("FAISS IVF/HNSW:  approximiert, hoeheres candidate_k erhoeht Recall.")

---

## 14. Hybrid Retrieval

`HybridRetriever` kombiniert Dense- und Sparse-Retriever via Fusion.

**HybridQuery:**
```python
hybrid_q: HybridQuery = {
    "dense":  List[float],                     # echter Embedding-Vektor
    "sparse": List[str] | Dict[str, float],    # tokenisiert oder learned sparse
}
```

`retrieve_with_details()` gibt das Tripel `(dense_candidates, sparse_candidates, fused_candidates)`
zurück. Das ist der Tracing-Pfad des Orchestrators.

In [ ]:
from rag.retrieval.hybrid import HybridRetriever
from rag.retrieval.config import FusionConfig

fusion_config = FusionConfig(
    strategy="weighted_sum",
    dense_weight=0.5,
    sparse_weight=0.5,
    combination="union",
    normalize_scores=True,
)

hybrid_retriever = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=bm25_retriever,
    fusion_config=fusion_config,
    dense_candidate_k=dense_config.candidate_k,
    sparse_candidate_k=bm25_config.candidate_k,
)

hybrid_query_str = "Retrieval Augmented Generation Sprachmodell"
hybrid_tokens    = tokenizer.tokenize(hybrid_query_str)

if EMBEDDER_AVAILABLE:
    hybrid_dense_vec = embed_query(hybrid_query_str)
    hybrid_source    = "echter Embedder"
else:
    hybrid_dense_vec = query_vec
    hybrid_source    = "Proxy-Vektor aus DocumentStore"

print(f"Query              : '{hybrid_query_str}'")
print(f"Dense-Vektor-Quelle: {hybrid_source}")
print(f"Dense-Dim          : {len(hybrid_dense_vec)} (= DENSE_DIM = {DENSE_DIM})")
print(f"Sparse-Tokens      : {hybrid_tokens}")

hybrid_q: HybridQuery = {"dense": hybrid_dense_vec, "sparse": hybrid_tokens}

d_cands, s_cands, fused_cands = hybrid_retriever.retrieve_with_details(query=hybrid_q, k=5)

print()
print(f"Dense candidates  : {len(d_cands)}")
print(f"Sparse candidates : {len(s_cands)}")
print(f"Fused candidates  : {len(fused_cands)}")
print()
print(f"{'ID (suffix)':<14} {'dense_score':>12}  {'sparse_score':>12}  {'fused_score':>11}")
print("-" * 56)
d_by_id = {c["id"]: c["score"] for c in d_cands}
s_by_id = {c["id"]: c["score"] for c in s_cands}
for fc in fused_cands:
    ds = d_by_id.get(fc["id"])
    ss = s_by_id.get(fc["id"])
    ds_str = f"{ds:+.4f}" if ds is not None else "         -"
    ss_str = f"{ss:+.4f}" if ss is not None else "         -"
    print(f"  {fc['id'][-12:]:<12}  {ds_str:>12}  {ss_str:>12}  {fc['score']:>11.5f}")

## 15. Fusion: Weighted Sum

`fuse()` kombiniert zwei `List[ScoredCandidate]` zu einer `List[RankedCandidate]`.

**Schritte:**
1. ID-Maps `{id: score}` für Dense und Sparse bilden
2. Kandidatenmenge bestimmen (`union` oder `intersection`)
3. optional pro Leg eine Min-Max-Normalisierung
4. gewichtete Summe: `w_dense * norm_dense + w_sparse * norm_sparse`
5. absteigend sortieren, auf k kürzen

**Warum Normalisierung entscheidend ist:** Dense-Scores (cosine) liegen in `[-1, 1]`,
BM25-Scores in `[0, 15+]`. Ohne Normalisierung dominiert BM25 allein wegen der größeren Zahlen.

**Zum Experimentieren:** `dense_weight` und `sparse_weight` verschieben das Gewicht zwischen
semantischer und lexikalischer Suche. `combination` entscheidet, ob nur gemeinsame Treffer
(`intersection`) oder alle Treffer (`union`) berücksichtigt werden. `normalize_scores`
schaltet die Normalisierung ein oder aus, der Effekt ist in der nächsten Zelle direkt
sichtbar. In Produktivsystemen werden diese Gewichte oft auf einem Evaluationsdatensatz
eingestellt statt geraten.


In [ ]:
from rag.retrieval.fusion import fuse

dense_ex  = [{"id": "doc-A", "score": 0.9}, {"id": "doc-B", "score": 0.7}, {"id": "doc-C", "score": 0.4}]
sparse_ex = [{"id": "doc-B", "score": 8.5}, {"id": "doc-D", "score": 6.0}, {"id": "doc-A", "score": 2.0}]

cfg_norm = FusionConfig(strategy="weighted_sum", dense_weight=0.5, sparse_weight=0.5,
                        normalize_scores=True, combination="union")
cfg_raw  = FusionConfig(strategy="weighted_sum", dense_weight=0.5, sparse_weight=0.5,
                        normalize_scores=False, combination="union")

result_norm = fuse(dense_ex, sparse_ex, cfg_norm, k=5)
result_raw  = fuse(dense_ex, sparse_ex, cfg_raw,  k=5)

print("Weighted Sum: normalize_scores=True vs. False")
print()
print(f"{'ID':<8}  {'norm=True':>9}  {'norm=False':>10}")
print("-" * 32)
norm_by_id = {r["id"]: (i+1, r["score"]) for i, r in enumerate(result_norm)}
raw_by_id  = {r["id"]: (i+1, r["score"]) for i, r in enumerate(result_raw)}
all_ids    = sorted(set(r["id"] for r in result_norm))
for did in all_ids:
    n_rank, n_score = norm_by_id.get(did, ("-", float("nan")))
    r_rank, r_score = raw_by_id.get(did, ("-", float("nan")))
    print(f"  {did:<6}  {n_score:>9.4f}  {r_score:>10.4f}")
print()
print("Ohne Normalisierung dominiert BM25 (doc-B score 8.5 >> cosine 0.7).")

## 16. Fusion: Reciprocal Rank Fusion (RRF)

RRF ist rang-basiert. Sie braucht keine Normalisierung, weil sie nie direkt mit Scores rechnet.

```
rrf_score(d) = Σ_r  1 / (rrf_k + rank_r(d))
```

**Warum RRF robust gegen inkompatible Score-Bereiche ist:** RRF betrachtet nur die Ränge.
Ein BM25-Score von 8.5 und ein cosine-Score von 0.9 sind für RRF gleichwertig, wenn beide in
ihrer jeweiligen Liste auf Rang 1 stehen.

**Zum Experimentieren:** `rrf_k` dämpft den Einfluss der vorderen Ränge. Kleines `rrf_k`
betont Rang 1 stark, großes `rrf_k` glättet die Unterschiede zwischen den Rängen. RRF ist eine
gute Wahl, wenn die Score-Bereiche der beiden Legs sehr unterschiedlich sind und eine
saubere Normalisierung schwerfällt. In Produktivsystemen ist RRF wegen dieser Robustheit ein
verbreiteter Standard für Hybrid-Fusion.


In [ ]:
dense_rrf  = [{"id": "A", "score": 0.95}, {"id": "B", "score": 0.80}, {"id": "C", "score": 0.60}]
sparse_rrf = [{"id": "C", "score": 12.0}, {"id": "A", "score":  9.5}, {"id": "B", "score":  4.0}]

cfg_rrf  = FusionConfig(strategy="rrf", rrf_k=60, combination="union")
cfg_wsum = FusionConfig(strategy="weighted_sum", dense_weight=0.5,
                        sparse_weight=0.5, normalize_scores=False, combination="union")

result_rrf  = fuse(dense_rrf, sparse_rrf, cfg_rrf, k=3)
result_wsum = fuse(dense_rrf, sparse_rrf, cfg_wsum, k=3)

print("RRF vs. Weighted Sum (normalize_scores=False)")
print("BM25-Scores ~10x groesser als cosine-Scores")
print()
print(f"{'ID':<4}  {'RRF Rang':>9}  {'RRF Score':>10}  {'WSum Rang':>10}  {'WSum Score':>10}")
print("-" * 50)
wsum_by_id = {r["id"]: (i+1, r["score"]) for i, r in enumerate(result_wsum)}
for i, r in enumerate(result_rrf):
    ws_rank, ws_score = wsum_by_id.get(r["id"], ("-", float("nan")))
    print(f"  {r['id']:<2}  {i+1:>9}  {r['score']:>10.5f}  {str(ws_rank):>10}  {ws_score:>10.4f}")
print()
print("RRF korrigiert die Verzerrung: C gewinnt (Rang-1 sparse), nicht A (hoechster cosine).")

---

## 17. DocumentStore-Join: Warum Text erst hier erscheint

`ScoredCandidate` und `RankedCandidate` enthalten nur IDs und Scores.
Text wird erst beim Join aus dem DocumentStore geladen.

**Warum der Join so spät passiert:**
Aus 100 Kandidaten werden vielleicht 5 gebraucht. Nur für diese 5 wird Text geladen.
Kein I/O für verworfene Kandidaten.

**Fehlerverhalten:**
Eine ID ohne DocumentStore-Eintrag wird übersprungen (Warning, kein Abbruch).

In [ ]:
def manual_join(candidates, doc_index):
    results = []
    for c in candidates:
        doc = doc_index.get(c["id"])
        if doc is None:
            print(f"  WARNING: id '{c['id']}' nicht im DocumentStore, uebersprungen")
            continue
        results.append({
            "id":              doc["id"],
            "chunk_id":        doc["chunk_id"],
            "document_id":     doc["document_id"],
            "score":           c["score"],
            "retrieval_score": c["score"],
            "text":            doc["text"],
            "metadata":        doc["metadata"],
        })
    results.sort(key=lambda r: (-r["score"], r["id"]))
    return results

joined = manual_join(dense_candidates[:3], doc_by_id)

# Fehlerhafter Fall
bad_cands  = [{"id": "emb-chunk-MISSING", "score": 0.99}] + dense_candidates[:2]
print("Fehlerhafter Fall (ID nicht vorhanden):")
bad_joined = manual_join(bad_cands, doc_by_id)
print(f"  -> {len(bad_joined)} Ergebnisse ({len(bad_cands) - len(bad_joined)} uebersprungen)")
print()

if joined:
    r = joined[0]
    print(f"Erstes RetrievalResult nach Join:")
    print(f"  score         : {r['score']:+.5f}")
    print(f"  text[:60]     : {r['text'][:60]!r}...")
    print(f"  metadata      : {r['metadata']}")

## 18. Reranking

`BaselineReranker` ist ein additiver Score-Anpasser nach dem Retrieval.

**Zwei Signale:**
```
lexical_overlap = |Q_tokens ∩ D_tokens| / |Q_tokens|   ∈ [0, 1]
length_score    = exp(-((len(text) - target)²) / (2*target²))  ∈ (0, 1]
```

**Score-Herkunft:**
```
rerank_delta           = lexical_weight * lexical_overlap + length_weight * length_score
result.score           = retrieval_score + rerank_delta
result.retrieval_score = Score vor dem Reranking (unverändert)
result.rerank_score    = rerank_delta (additiver Beitrag des Rerankers)
```

Reranking verändert die Kandidatenmenge nicht, nur ihre Reihenfolge.

**Zum Experimentieren:** `lexical_weight` und `length_weight` steuern, wie stark
Wortüberlappung und Ziel-Länge die Reihenfolge verschieben. `target_length` legt fest, welche
Chunk-Länge als ideal gilt. Mit `enabled=False` lässt sich der Reranker ganz abschalten und
der Effekt am Ranking ablesen. In Produktivsystemen wird dieser einfache, regelbasierte
Reranker oft durch ein Cross-Encoder-Modell ersetzt, das Query und Dokument gemeinsam bewertet.


In [ ]:
from rag.retrieval.reranker import BaselineReranker
from rag.retrieval.config import RerankerConfig

reranker_cfg = RerankerConfig(enabled=True, lexical_weight=0.5, length_weight=0.5, target_length=100)
reranker     = BaselineReranker(reranker_cfg)

reranked = reranker.rerank(query=hybrid_query_str, results=joined)

print(f"Query: '{hybrid_query_str}'")
print()
print(f"{'Rang':<5} {'retrieval_score':>16}  {'rerank_delta':>12}  {'final_score':>11}  Text[:40]")
print("-" * 82)
for i, r in enumerate(reranked):
    print(f"  {i+1:<3}  {r['retrieval_score']:>16.5f}  "
          f"{r.get('rerank_score', 0.0):>12.4f}  "
          f"{r['score']:>11.5f}  {r['text'][:40]!r}...")

## 19. RetrievalOrchestrator: Runtime-Steuerung

Der `RetrievalOrchestrator` verbindet die vollständige Online-Pipeline: Query-Transformation,
Candidate Generation, Scoring, Fusion, DocumentStore-Join und Reranking laufen nacheinander ab.

`retrieve_with_trace()` gibt zusätzlich alle Zwischenschritte zurück, was die Pipeline
nachvollziehbar macht.


In [ ]:
from rag.retrieval.orchestrator import RetrievalOrchestrator
from rag.retrieval.config import (
    RetrievalConfig, BM25Config, TFIDFConfig,
    DenseRetrievalConfig, FusionConfig, RerankerConfig,
)
from rag.retrieval.types import RetrievalTrace

retrieval_config = RetrievalConfig(
    mode="hybrid",
    sparse_strategy="bm25",
    top_k=5,
    dense=DenseRetrievalConfig(candidate_k=10),
    bm25=BM25Config(k1=1.5, b=0.75, candidate_k=10),
    fusion=FusionConfig(
        strategy="weighted_sum",
        dense_weight=0.5,
        sparse_weight=0.5,
        normalize_scores=True,
        combination="union",
    ),
    reranker=RerankerConfig(enabled=False),
)

if not EMBEDDER_AVAILABLE:
    raise RuntimeError(
        "RetrievalOrchestrator im Hybrid-Modus erfordert einen echten Embedder. "
        "Bitte das Embedding-Modell installieren oder mode='sparse_bm25' verwenden."
    )

orchestrator = RetrievalOrchestrator(
    document_store=doc_store,
    config=retrieval_config,
    embedder=embedder,
    dense_index=dense_index,
    sparse_index=sparse_index,
    tokenizer=tokenizer,
)

print("RetrievalOrchestrator initialisiert:")
print(f"  mode            : {retrieval_config.mode}")
print(f"  sparse_strategy : {retrieval_config.sparse_strategy}")
print(f"  top_k           : {retrieval_config.top_k}")
print(f"  dense backend   : {type(dense_index).__name__}")
print(f"  sparse backend  : {type(sparse_index).__name__}")
print(f"  embedder        : {embedder.model_type()}")
print(f"  documents       : {len(all_docs)}")

In [ ]:
# retrieve_with_trace(): vollständige Intermediate States
trace_query = "BM25 Rankingmodell Textretrieval"
trace: RetrievalTrace = orchestrator.retrieve_with_trace(query=trace_query, k=4)

print(f"=== RetrievalTrace ===")
print(f"mode              : {trace['mode']}")
print(f"query             : '{trace['query']}'")
print(f"query_tokens      : {trace['query_tokens']}")
print()
print(f"dense_candidates  : {len(trace['dense_candidates'])}")
print(f"sparse_candidates : {len(trace['sparse_candidates'])}")
print(f"fused_candidates  : {len(trace['fused_candidates'])}")
print(f"final_results     : {len(trace['final_results'])}")

In [ ]:
print("Dense-Kandidaten:")
for c in trace["dense_candidates"][:3]:
    print(f"  {c['id'][-12:]:<14}  score={c['score']:+.5f}")
print()
print("Sparse (BM25) Kandidaten:")
for c in trace["sparse_candidates"][:3]:
    print(f"  {c['id'][-12:]:<14}  score={c['score']:.5f}")
print()
print("Fused Kandidaten (weighted_sum, normalize=True):")
for c in trace["fused_candidates"]:
    ds = f"{c['dense_score']:+.4f}"  if c.get("dense_score") is not None else "      -"
    ss = f"{c['sparse_score']:.4f}"  if c.get("sparse_score") is not None else "      -"
    print(f"  {c['id'][-12:]:<14}  dense={ds}  sparse={ss}  fused={c['score']:.5f}")

In [ ]:
print("Finale RetrievalResult-Objekte:")
print()
for i, r in enumerate(trace["final_results"]):
    print(f"  [{i+1}] score={r['score']:.5f}  retrieval_score={r['retrieval_score']:.5f}")
    print(f"       text: {r['text'][:65]!r}...")
    print()

### RetrievalTrace: Analyse der Erklärbarkeit

Mit `RetrievalTrace` werden konkrete Fragen beantwortbar:

- **Warum steht Dokument X auf Rang 1?** Die `fused_candidates` zeigen den fusionierten Score.
- **Welches Leg hat stärker beigetragen?** Der Vergleich von `dense_score` und `sparse_score`.
- **Welche Dokumente hat nur Dense gefunden?** Die Differenz der Kandidatenmengen
  (`dense_ids` ohne `sparse_ids`).


In [ ]:
dense_ids  = {c["id"] for c in trace["dense_candidates"]}
sparse_ids = {c["id"] for c in trace["sparse_candidates"]}

only_dense  = dense_ids - sparse_ids
only_sparse = sparse_ids - dense_ids
both        = dense_ids & sparse_ids

print("Kandidaten-Analyse:")
print(f"  Nur Dense     : {len(only_dense)}")
print(f"  Nur Sparse    : {len(only_sparse)}")
print(f"  Beide Legs    : {len(both)}")
print()

print("Dokumente exklusiv im Dense-Leg:")
for eid in sorted(only_dense):
    doc = doc_by_id.get(eid)
    print(f"  {eid[-14:]:<16}  {doc['text'][:48]!r}..." if doc else f"  {eid}")
print()
print("Dokumente exklusiv im Sparse-Leg:")
for eid in sorted(only_sparse):
    doc = doc_by_id.get(eid)
    print(f"  {eid[-14:]:<16}  {doc['text'][:48]!r}..." if doc else f"  {eid}")

---

## 20. Architektur-Invarianten

Die folgenden Invarianten gelten für alle Betriebszustände.

In [ ]:
# 1. DenseRetriever gibt niemals Text zurück
for c in dense_retriever.retrieve_candidates(query=query_vec, k=5):
    assert "text" not in c
print("[OK] DenseRetriever: kein 'text' in ScoredCandidate")

# 2. BM25Retriever gibt niemals Text zurück
for c in bm25_retriever.retrieve_candidates(query="Retrieval", k=5):
    assert "text" not in c
print("[OK] BM25Retriever: kein 'text' in ScoredCandidate")

# 3. SparseIndex scored nicht selbst
raw_sparse = sparse_index.query(["retrieval", "sparse"], k=5)
for c in raw_sparse:
    assert "score" not in c
print("[OK] SparseIndex: kein 'score' in SparseCandidate")

# 4. Deterministisch sortiert: Score DESC, id ASC
candidates = dense_retriever.retrieve_candidates(query=query_vec, k=8)
for i in range(len(candidates) - 1):
    a, b = candidates[i], candidates[i+1]
    assert (a["score"] > b["score"]) or (a["score"] == b["score"] and a["id"] <= b["id"])
print("[OK] DenseRetriever: deterministisch sortiert (Score DESC, id ASC)")

# 5. RetrievalResult hat immer retrieval_score und text
for r in trace["final_results"]:
    assert "retrieval_score" in r
    assert "text" in r and r["text"]
print("[OK] RetrievalResult: retrieval_score und text immer vorhanden")

# 6. min_max_normalise: gleiche Scores -> 1.0
eq = min_max_normalise([3.0, 3.0, 3.0])
assert eq == [1.0, 1.0, 1.0]
print("[OK] min_max_normalise: gleiche Scores -> [1.0, 1.0, 1.0]")

# 7. Query-Dimension muss DENSE_DIM entsprechen
assert len(query_vec) == DENSE_DIM
print(f"[OK] query_vec hat korrekte Dimension: {DENSE_DIM}")

# 8. Alle gespeicherten Dense-Vektoren haben dieselbe Dimension
stored_dims = {len(d["dense_vector"]) for d in docs_with_dense}
assert len(stored_dims) == 1 and next(iter(stored_dims)) == DENSE_DIM
print(f"[OK] Alle Dense-Vektoren im DocumentStore: dim={DENSE_DIM}")

print()
print("Alle Architektur-Invarianten bestaetigt.")

---

## 21. Bereitschaftsprüfung des Retrieval

In [ ]:
all_docs_final  = list(doc_store.stream_all())
doc_ids_final   = {d["id"] for d in all_docs_final}
dense_test      = dense_index.query(query_vec, k=3)
sparse_test     = sparse_index.query(["retrieval"], k=3)
sparse_stats_f  = sparse_index.get_view().get_stats()
tokenizer_test  = tokenizer.tokenize("BM25 Retrieval Modell")

stored_dims_f  = {len(d["dense_vector"]) for d in all_docs_final if d.get("dense_vector")}
dim_consistent = len(stored_dims_f) == 1 and next(iter(stored_dims_f)) == DENSE_DIM
emb_dim_ok     = (not EMBEDDER_AVAILABLE) or (embedder.dimension() is None) or (embedder.dimension() == DENSE_DIM)

readiness = {
    "Artefakte geladen":              dense_index is not None and sparse_index is not None,
    "DocumentStore nicht leer":       len(all_docs_final) > 0,
    "IDs eindeutig":                  len(doc_ids_final) == len(all_docs_final),
    "Texte nicht leer":               all(d.get("text", "").strip() for d in all_docs_final),
    "Dense-Dimension konsistent":     dim_consistent,
    "Dense queryable":                isinstance(dense_test, list),
    "Dense IDs im DocumentStore":     all(r["id"] in doc_ids_final for r in dense_test),
    "Sparse queryable":               isinstance(sparse_test, list),
    "Sparse stats konsistent":        sparse_stats_f.get("doc_count", -1) == len(all_docs_final),
    "Tokenizer aktiv":                len(tokenizer_test) > 0,
    "Embedder Dimension kompatibel":  emb_dim_ok,
}

for label, ok in readiness.items():
    print(f"[{'OK' if ok else 'FAIL'}] {label}")

if not all(readiness.values()):
    raise RuntimeError("Retrieval-Readiness-Check fehlgeschlagen.")

v = sparse_index.get_view()
print()
print("RETRIEVAL-READY")
print(f"  Dokumente    : {len(all_docs_final)}")
print(f"  Dense dim    : {DENSE_DIM}")
print(f"  Vocabulary   : {len(v.get_vocabulary())} Terme")
print(f"  Modus        : {retrieval_config.mode}")
print(f"  Embedder     : {embedder.model_type() if EMBEDDER_AVAILABLE else 'nicht verfuegbar'}")

## 22. Zusammenfassung: Was die Retrieval Layer leistet

### Was hier gezeigt wurde

**Echte Artefakte:** Alle persistierten Indizes wurden aus dem Indexing-Notebook geladen.

**Echter Embedder:** Query-Vektoren für Dense-Retrieval werden vom echten Embedding-Modell
erzeugt, mit derselben Konfiguration wie beim Index-Aufbau. Ein Dimensionsmismatch wird sofort
erkannt und führt zum Abbruch.

**Proxy-Vektoren für Isolation:** Wo ein echter Embedder-Aufruf für eine einzelne
Demonstrationszelle zu teuer wäre, dienen echte, persistierte Dokumentvektoren aus dem
DocumentStore als Query. Diese Vektoren sind real und haben die korrekte Dimension.

### Architektur-Invarianten

| Invariante | Implikation |
|---|---|
| DenseRetriever gibt niemals Text zurück | Text liegt ausschließlich im DocumentStore |
| SparseIndex scored nicht | Scoring ist query-abhängig und gehört zur Retrieval Layer |
| Query-Dimension gleich Index-Dimension | ein Mismatch ist ein harter Fehler |
| Fusion vor dem Join | Join-Kosten fallen nur für finale Kandidaten an |
| Reranking nach dem Join | lexikalischer Overlap braucht den Dokumenttext |
| Sortierung deterministisch | Score absteigend, `id` aufsteigend, reproduzierbar |
| min_max_normalise bei gleichen Scores ergibt 1.0 | kein Leg fällt durch Normalisierung still heraus |
| Retrieval ist zustandslos | jede Query ist unabhängig, keine Seiteneffekte |

### Datenfluss der vollständigen Pipeline bis hierher

Offline (einmalig gebaut):

1. Rohdaten, Ingestion Layer, ergibt `chunks.jsonl`
2. Embedding Layer, ergibt `embeddings.jsonl` (echte Modellvektoren)
3. Indexing Layer, ergibt DenseIndex, SparseIndex und DocumentStore

Online (pro Query, hinter der Offline-Grenze):

1. `embed_queries()` erzeugt den Dense-Vektor (Dimension `DENSE_DIM`)
2. `tokenize()` erzeugt die Sparse-Tokens
3. `DenseIndex.query()` und `SparseIndex.query()` liefern Kandidaten
4. BM25 berechnet die Sparse-Scores
5. `fuse()` führt beide Listen zu `RankedCandidate` zusammen
6. der DocumentStore-Join macht daraus `RetrievalResult` mit Text
7. der `BaselineReranker` bewertet die oberen Ränge nach

Die nächste Stufe ist die Generation Layer.
